# Conversational Memory Assistant

**Mode:** Offline  
**Level:** Capstone  
**Estimated time:** 35 minutes

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## Problem and outcome

Build an offline assistant that remembers stable preferences, records important events, keeps recent working context, and explains which memory influenced each answer. The outcome is a personalized response with inspectable memory boundaries.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(), Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    get_events, require_env, setup_case_study, show_callout,
    show_flow, show_json, show_roles, show_spore, show_timeline, stage,
)

setup_case_study('Conversational Memory Assistant')


## Course prerequisites

This capstone assumes: `course-06-agent-memory`. The implementation focuses on system design instead of explaining routine Python syntax.


## Architecture and responsibilities

The conversation layer classifies incoming facts before storage. The Agent's MemoryManager owns typed entries and retrieval. The response composer uses only the returned entries and records an explanation trail.


In [ ]:
show_roles([
('Conversation', 'Facts and questions', 'human'),
('Memory policy', 'Classify and store', 'memory'),
('Recall', 'Retrieve scoped context', 'agent'),
('Composer', 'Answer with evidence', 'agent')
])


## Design decisions

- Preferences become semantic memory; completed interactions become episodic memory.
- Transient user wording remains short-term instead of being promoted automatically.
- Answers expose the memory IDs and types they used.
- The capstone stays deterministic so memory decisions, not model prose, are evaluated.


## Implementation

The next cells are grouped by subsystem. Each group exposes the Praval objects that matter to the architecture.


In [ ]:
from praval import Agent

assistant = Agent(
    "case-memory-assistant", provider="notebook-lifecycle",
    model="notebook-lifecycle-model", memory_enabled=True,
    memory_config={"backend": "memory"},
)
assert assistant.memory is not None
conversation_log = []


### Memory policy

Only explicit, useful facts are promoted beyond working context.


In [ ]:
def record_fact(content, kind, importance):
    allowed = {"short_term", "semantic", "episodic"}
    if kind not in allowed:
        raise ValueError(f"unsupported memory boundary: {kind}")
    memory_id = assistant.remember(
        content, importance=importance, memory_type=kind
    )
    assert memory_id
    conversation_log.append({
        "action": "remember", "id": memory_id, "type": kind,
        "content": content,
    })
    return memory_id


preference_id = record_fact(
    "The user prefers concise technical explanations.", "semantic", 0.95
)
event_id = record_fact(
    "The user completed the Reef routing lesson.", "episodic", 0.8
)
working_id = record_fact(
    "The current question is about the next lesson.", "short_term", 0.4
)


### Response policy

Retrieval evidence is part of the result, not hidden prompt state.


In [ ]:
def answer(question):
    recalled = assistant.recall(
        question, limit=5, similarity_threshold=0.0
    )
    used = [
        {"id": item.id, "type": item.memory_type.value,
         "content": item.content}
        for item in recalled
    ]
    concise = any("concise" in item["content"] for item in used)
    completed_reef = any("Reef" in item["content"] for item in used)
    recommendation = (
        "Continue with Spores and pipelines."
        if completed_reef else "Begin with Reef routing."
    )
    response = recommendation if concise else recommendation + " Review the examples."
    result = {"question": question, "answer": response, "memory_used": used}
    conversation_log.append({"action": "answer", **result})
    return result


## End-to-end run

Ask one personalization question and assert both the recommendation and evidence.


In [ ]:
result = answer("What should I learn next, given my preferences and progress?")
assert result["answer"] == "Continue with Spores and pipelines."
used_ids = {item["id"] for item in result["memory_used"]}
assert preference_id in used_ids and event_id in used_ids
stage("assistant", "personalized answer", result["answer"])
show_json(result, "Personalized response with memory evidence")


## Inspect the system

Inspect the evidence left by the completed run.


In [ ]:
show_json(conversation_log, "Conversation and memory decision trail")
show_json(assistant.memory.get_memory_stats(), "Memory subsystem statistics")
show_json(
    {
        "preference": assistant.recall_by_id(preference_id)[0].memory_type.value,
        "event": assistant.recall_by_id(event_id)[0].memory_type.value,
        "working": assistant.recall_by_id(working_id)[0].memory_type.value,
    },
    "Verified memory boundaries",
)


## Failure modes and tradeoffs

- The in-memory backend does not survive process exit; Qdrant or another persistent store is needed for durable personalization.
- Naive keyword decisions are transparent but limited; a learned policy needs evaluation and privacy review.
- Recall may return more context than an answer needs, so prompt budgets and access controls still matter.
- Incorrect preferences need correction and deletion paths, not only append-only storage.


## Extensions

- Add a correction command that replaces an obsolete preference and proves the old one is no longer used.
- Move semantic and episodic memory to Qdrant while keeping working context local.
- Add a privacy label to memories and filter retrieval before response composition.


## Cleanup

Release project resources explicitly.


In [ ]:
assistant.memory.clear_agent_memories(assistant.name)
assistant.close()
assert assistant._closed
